# Enron Email Topic Modelling Project — revised final notebook

## Research question
What are the main themes in employee communication in the Enron email corpus?

## Technical goal
Build a topic-modelling pipeline that identifies **interpretable and defensible** communication themes rather than noisy document formats.

## What this revised notebook changes
- keeps the stronger structure of the previous `90plus` version;
- removes several **residual noisy document families** found in the previous output review;
- avoids hard-coding the final number of topics too early;
- makes the final model-selection logic more explicit;
- leaves clear `???` placeholders where the written interpretation should be completed **after** re-running the notebook and reviewing the final output.

## Important note
This notebook should be defended as an **iterative modelling project**:
1. clean the corpus;
2. compare several topic counts;
3. inspect representative emails;
4. choose the most interpretable solution;
5. write the final interpretation only after the evidence is visible.

## 1. Imports and configuration


In [1]:
import os
import re
import html
import math
import random
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.metrics.pairwise import cosine_similarity

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from gensim.models.phrases import Phrases, Phraser

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

pd.set_option("display.max_colwidth", 300)


## 2. Load the dataset


In [2]:
# Option A: use the Kaggle source, like in the original notebook
USE_KAGGLEHUB = True

if USE_KAGGLEHUB:
    import kagglehub

    dataset_path = kagglehub.dataset_download("wcukierski/enron-email-dataset")
    csv_path = os.path.join(dataset_path, "emails.csv")
else:
    # Option B: manual local path fallback
    csv_path = "emails.csv"

df = pd.read_csv(csv_path)

print("Dataset loaded.")
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

# Keep only the text column that matters
if "message" not in df.columns:
    raise ValueError("Expected a 'message' column in the dataset.")

df = df[["message"]].copy()
df.head(3)


Dataset loaded.
Rows: 517401
Columns: ['file', 'message']


,message
0,"Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>\nDate: Mon, 14 May 2001 16:39:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: tim.belden@enron.com\nSubject: \nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nX-From: Phillip K Allen\nX-T..."
1,"Message-ID: <15464986.1075855378456.JavaMail.evans@thyme>\nDate: Fri, 4 May 2001 13:51:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: john.lavorato@enron.com\nSubject: Re:\nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nX-From: Phillip K Allen..."
2,"Message-ID: <24216240.1075855687451.JavaMail.evans@thyme>\nDate: Wed, 18 Oct 2000 03:00:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: leah.arsdall@enron.com\nSubject: Re: test\nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nX-From: Phillip K ..."


## 3. Why preprocessing matters in email topic modelling

The raw Enron corpus contains many documents that are **not genuine conversational content** in the narrow sense needed for topic modelling.  
Examples include forwarded chains, legal boilerplate, admin notices, travel confirmations, newsletters, system logs, and repeated tabular reports.

So the quality of the final topics depends less on “using a fancy model” and more on building a corpus that actually contains meaningful communication signals.

## 4. Cleaning functions


In [3]:
HEADER_PREFIXES = (
    "message-id:", "date:", "from:", "to:", "cc:", "bcc:", "subject:",
    "mime-version:", "content-type:", "content-transfer-encoding:",
    "x-from:", "x-to:", "x-cc:", "x-bcc:", "x-folder:", "x-origin:",
    "x-filename:"
)

REPLY_CHAIN_MARKERS = [
    r"(?im)^-----original message-----",
    r"(?im)^from:\s",
    r"(?im)^sent:\s",
    r"(?im)^to:\s",
    r"(?im)^subject:\s",
    r"(?im)^on .+ wrote:$",
    r"(?im)^forwarded by .*",
]

BOILERPLATE_PATTERNS = [
    # confidentiality / legal footer boilerplate
    r"(?is)this e-?mail and any attachments?.*?intended recipient.*",
    r"(?is)if you are not the intended recipient.*",
    r"(?is)delete this message.*",
    r"(?is)please notify the sender immediately.*",
    r"(?is)any review, use, disclosure.*",
    r"(?is)virus warning.*",

    # html / web junk
    r"(?is)<[^>]+>",
    r"(?is)&nbsp;",
    r"(?is)http[s]?://\S+",
    r"(?is)www\.\S+",

    # attachment / system noise
    r"(?im)^\s*\[image\].*$",
    r"(?im)^\s*inline attachment follows.*$",
    r"(?im)^\s*start date:.*$",
    r"(?im)^\s*end date:.*$",
    r"(?im)^\s*location:.*$",
    r"(?im)^\s*attachments?:.*$",
]

NOISE_DOC_PATTERNS = [
    # newsletters / promos / subscriptions
    r"thank you for subscribing",
    r"unsubscribe",
    r"click here",
    r"view this email",
    r"special offer",
    r"web specials",
    r"dear subscriber",
    r"mailing list",
    r"membership services",

    # calendar / survey / admin
    r"calendar entry",
    r"questionnaire",
    r"please complete",
    r"survey",
    r"outlook migration",
    r"distribution groups",
    r"shared calendar",

    # travel
    r"travel itinerary",
    r"flight summary",
    r"boarding pass",
    r"economy coach class",
    r"depart houston",
    r"arrive houston",

    # chain / greeting / event spam
    r"please pass this message along",
    r"joyful holiday season",
    r"the more the merrier",
    r"hope everyone can make it",
    r"weekend we will be hosting",
]

# Residual noisy families found during qualitative review of the previous run
RESIDUAL_FAMILY_PATTERNS = [
    # system / schedule variance logs
    r"variances detected",
    r"schedule variances",
    r"log messages parsing file",
    r"error retrieving hourahead price data",
    r"cannot locate a preferred or revised preferred schedule",
    r"hour bad data from iso",

    # repeated capacity / allocation tables
    r"working capacity allocation",
    r"quantity cut cd",
    r"receipt delivery quantity",
    r"confirmed cbl",
    r"confirmed ccr",

    # repeated trade summary tables
    r"total trades for day",
    r"desk total deals total mwh",
    r"epmi long term california",
    r"epmi short term california",
    r"grand total grand total",

    # petition / advocacy duplicates
    r"writing to urge you to donate",
    r"their retirement savings",
    r"are unable to pay their energy bills",
    r"repair the lives of those americans hurt",
    r"company declared bankruptcy",
]

def normalize_newlines(text):
    text = str(text).replace("\r\n", "\n").replace("\r", "\n")
    text = html.unescape(text)
    return text

def remove_top_header(text):
    text = normalize_newlines(text).strip()
    if not text:
        return ""

    m = re.search(r'(?im)^X-FileName:.*$\n?', text)
    if m:
        return text[m.end():].strip()

    lines = text.split("\n")
    body_start = 0

    for i, line in enumerate(lines):
        stripped = line.strip()
        if stripped == "":
            body_start = i + 1
            break

        if stripped.startswith(HEADER_PREFIXES):
            continue

        body_start = i
        break

    return "\n".join(lines[body_start:]).strip()

def cut_reply_chain(text):
    text = normalize_newlines(text).strip()
    if not text:
        return ""

    cut_positions = []
    for pattern in REPLY_CHAIN_MARKERS:
        m = re.search(pattern, text)
        if m:
            cut_positions.append(m.start())

    if cut_positions:
        text = text[:min(cut_positions)]

    return text.strip()

def remove_boilerplate(text):
    text = normalize_newlines(text)
    for pattern in BOILERPLATE_PATTERNS:
        text = re.sub(pattern, " ", text)
    return re.sub(r"\s+", " ", text).strip()

def remove_signature(text):
    text = normalize_newlines(text).strip()
    if not text:
        return ""

    signature_markers = [
        r"(?im)^thanks,?$",
        r"(?im)^thank you,?$",
        r"(?im)^regards,?$",
        r"(?im)^best,?$",
        r"(?im)^best regards,?$",
        r"(?im)^sincerely,?$",
        r"(?im)^cordially,?$",
        r"(?im)^sent from my .*",
    ]

    lines = text.split("\n")
    for i, line in enumerate(lines):
        stripped = line.strip()
        if any(re.match(p, stripped) for p in signature_markers):
            return "\n".join(lines[:i]).strip()

    return text

def normalize_for_model(text):
    text = normalize_newlines(text).lower()

    # emails, urls, numbers, punctuation
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\b\d+[\d\-/.:]*\b', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

def full_clean(text):
    x = remove_top_header(text)
    x = cut_reply_chain(x)
    x = remove_boilerplate(x)
    x = remove_signature(x)
    x = normalize_for_model(x)
    return x

def is_noise_document(clean_text):
    if not clean_text:
        return True
    return bool(re.search("|".join(NOISE_DOC_PATTERNS), clean_text, flags=re.I))

def is_residual_family_document(clean_text):
    if not clean_text:
        return False
    return bool(re.search("|".join(RESIDUAL_FAMILY_PATTERNS), clean_text, flags=re.I))

def text_signature(text):
    # For near-duplicate removal: remove very weak variation
    text = re.sub(r'\b(?:monday|tuesday|wednesday|thursday|friday|saturday|sunday)\b', ' ', text)
    text = re.sub(r'\b(?:january|february|march|april|may|june|july|august|september|october|november|december)\b', ' ', text)
    text = re.sub(r'\b[a-z]{1,2}\b', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def lexical_diversity(text):
    tokens = str(text).split()
    if not tokens:
        return 0.0
    return len(set(tokens)) / len(tokens)

def max_token_share(text):
    tokens = str(text).split()
    if not tokens:
        return 1.0
    counts = Counter(tokens)
    return max(counts.values()) / len(tokens)

## 5. Apply the cleaning pipeline

At this stage we generate the core cleaned text and a few simple document-quality diagnostics.
These diagnostics are later used to remove documents that are too short, too repetitive, or too dominated by template structure.

In [4]:
df["clean_text"] = df["message"].apply(full_clean)
df["word_len"] = df["clean_text"].str.split().str.len()
df["noise_flag"] = df["clean_text"].apply(is_noise_document)
df["residual_family_flag"] = df["clean_text"].apply(is_residual_family_document)
df["lexical_diversity"] = df["clean_text"].apply(lexical_diversity)
df["max_token_share"] = df["clean_text"].apply(max_token_share)

print("Original rows:", len(df))
print(df["word_len"].describe())
print("Noise-flagged rows:", int(df["noise_flag"].sum()))
print("Residual-family rows:", int(df["residual_family_flag"].sum()))
print("Median lexical diversity:", round(df["lexical_diversity"].median(), 3))
print("Median max token share:", round(df["max_token_share"].median(), 3))

df[["message", "clean_text", "word_len", "lexical_diversity", "max_token_share"]].head(5)

Original rows: 517401
count    517401.000000
mean        155.988025
std         750.483329
min           0.000000
25%          16.000000
50%          41.000000
75%         113.000000
max       63483.000000
Name: word_len, dtype: float64
Noise-flagged rows: 38312
Residual-family rows: 9567
Median lexical diversity: 0.824
Median max token share: 0.073


,message,clean_text,word_len,lexical_diversity,max_token_share
0,"Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>\nDate: Mon, 14 May 2001 16:39:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: tim.belden@enron.com\nSubject: \nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nX-From: Phillip K Allen\nX-T...",here is our forecast,4,1.000000,0.250000
1,"Message-ID: <15464986.1075855378456.JavaMail.evans@thyme>\nDate: Fri, 4 May 2001 13:51:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: john.lavorato@enron.com\nSubject: Re:\nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nX-From: Phillip K Allen...",traveling to have a business meeting takes the fun out of the trip especially if you have to prepare a presentation i would suggest holding the business plan meetings here then take a trip without any formal business meetings i would even try and get some honest opinions on whether a trip is eve...,140,0.671429,0.057143
2,"Message-ID: <24216240.1075855687451.JavaMail.evans@thyme>\nDate: Wed, 18 Oct 2000 03:00:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: leah.arsdall@enron.com\nSubject: Re: test\nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nX-From: Phillip K ...",test successful way to go,5,1.000000,0.200000
3,"Message-ID: <13505866.1075863688222.JavaMail.evans@thyme>\nDate: Mon, 23 Oct 2000 06:13:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: randall.gay@enron.com\nSubject: \nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nX-From: Phillip K Allen\nX-...",randy can you send me a schedule of the salary and level of everyone in the scheduling group plus your thoughts on any changes that need to be made patti s for example phillip,34,0.941176,0.058824
4,"Message-ID: <30922949.1075863688243.JavaMail.evans@thyme>\nDate: Thu, 31 Aug 2000 05:07:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: greg.piper@enron.com\nSubject: Re: Hello\nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nX-From: Phillip K A...",let s shoot for tuesday at,6,1.000000,0.166667


## 6. Filter obvious low-quality documents

We keep documents that are long enough to contain a topic signal and that do not look like obvious noise.

In the revised version, we also use simple repetition indicators because the previous output still contained several highly repetitive table-like documents.

In [5]:
df_model = df[
    (df["word_len"] >= 25) &
    (df["word_len"] <= 1200) &
    (~df["noise_flag"]) &
    (~df["residual_family_flag"]) &
    (df["lexical_diversity"] >= 0.22) &
    (df["max_token_share"] <= 0.12)
].copy()

print("Rows after basic filtering:", len(df_model))
print("Rows removed by residual family filter:", int(df["residual_family_flag"].sum()))

Rows after basic filtering: 287956
Rows removed by residual family filter: 9567


## 7. Exact and near-duplicate filtering

This step removes both exact duplicate emails and very close near-duplicates after weak textual variation has been normalized.
That is especially important in email corpora, where repeated templates can dominate whole topics.

In [6]:
# Exact duplicates
before_exact = len(df_model)
df_model = df_model.drop_duplicates(subset=["clean_text"]).copy()
after_exact = len(df_model)

# Near duplicates using normalized signature
df_model["signature_text"] = df_model["clean_text"].apply(text_signature)
before_near = len(df_model)
df_model = df_model.drop_duplicates(subset=["signature_text"]).copy()
after_near = len(df_model)

print("Before duplicate filtering:", before_exact)
print("After exact duplicate removal:", after_exact)
print("After near-duplicate removal:", after_near)
print("Exact duplicates removed:", before_exact - after_exact)
print("Near duplicates removed:", after_exact - after_near)

df_model = df_model.drop(columns=["signature_text"])


Before duplicate filtering: 287956
After exact duplicate removal: 124645
After near-duplicate removal: 124060
Exact duplicates removed: 163311
Near duplicates removed: 585


## 8. Stronger modelling subset

After the basic filters, duplicate removal, and residual-family control, we keep only documents that are long enough to contain a real topic signal.
Very short emails tend to produce weak clusters such as `thanks / call / tomorrow / let know`.

In [7]:
df_topics = df_model[
    (df_model["word_len"] >= 40) &
    (df_model["word_len"] <= 900)
].copy()

docs_raw = df_topics["clean_text"].dropna().astype(str)
docs_raw = docs_raw[docs_raw.str.strip() != ""].reset_index(drop=True)

print("Documents used for modelling:", len(docs_raw))
docs_raw.head()

Documents used for modelling: 95480


0    traveling to have a business meeting takes the fun out of the trip especially if you have to prepare a presentation i would suggest holding the business plan meetings here then take a trip without any formal business meetings i would even try and get some honest opinions on whether a trip is eve...
1                                           lucy here are the rentrolls open them and save in the rentroll folder follow these steps so you don t misplace these files click on save as click on the drop down triangle under save in click on the c drive click on the appropriate folder click on save phillip
2    forwarded by phillip k allen hou ect on pm from phillip k allen am liane as we discussed yesterday i am concerned there may have been an attempt to manipulate the el paso san juan monthly index it appears that a single buyer entered the marketplace on both september and and paid above market pri...
3    liane as we discussed yesterday i am concerned there has been an attempt to mani

## 9. Tokenization and phrase detection

Phrase detection helps preserve business expressions such as `credit risk`, `master agreement`, or `gas price` as more meaningful units.
That usually improves interpretability compared with treating every word independently.

In [8]:
def simple_tokenize(text):
    return [tok for tok in text.split() if len(tok) >= 3]

tokenized_docs = [simple_tokenize(doc) for doc in docs_raw]

# Detect common bigrams in the corpus
phrases = Phrases(tokenized_docs, min_count=20, threshold=20)
bigram = Phraser(phrases)

tokenized_docs_phrased = [bigram[doc] for doc in tokenized_docs]

# Rebuild text after phrase detection
docs_phrased = pd.Series([" ".join(doc) for doc in tokenized_docs_phrased])

print("Sample with phrases:")
docs_phrased.head(5)


Sample with phrases:


0    traveling have business meeting takes the fun out the trip especially you have prepare presentation would suggest holding the business plan meetings here then take trip without any formal business meetings would even try and get some honest opinions whether trip even desired necessary far the bu...
1                                                                          lucy here are the rentrolls open them and save the rentroll folder follow these steps you don misplace these files click save click the drop_down triangle under save click the drive click the appropriate folder click save phillip
2    forwarded phillip_allen hou_ect from phillip_allen liane discussed yesterday concerned there may have been attempt manipulate the paso san_juan monthly index appears that single buyer entered the marketplace both september and and paid above market prices for san_juan gas the time these trades o...
3    liane discussed yesterday concerned there has_been attempt manipulate the paso s

## 10. Data-driven stopwords


In [9]:
base_stopwords = set(ENGLISH_STOP_WORDS)

domain_stopwords = {
    "enron", "ect", "hou", "corp", "com", "email", "mail", "message",
    "subject", "cc", "bcc", "fwd", "fw", "re", "forwarded",
    "attached", "attachment", "original", "please", "thanks", "thank",
    "regards", "best", "let", "know", "need", "sent", "said", "today",
    "tomorrow", "yesterday", "ll", "ve", "don", "didn", "doesn",
    "phillip", "allen", "john", "randy", "david", "dave", "mike",
    "ina", "julie", "paula", "brenda", "cynthia",
}

# Add very frequent weak terms from the corpus
all_tokens = [tok for doc in tokenized_docs_phrased for tok in doc]
freq = Counter(all_tokens)

very_frequent_terms = {
    tok for tok, count in freq.items()
    if count > 0.30 * len(tokenized_docs_phrased) and "_" not in tok
}

final_stopwords = sorted(base_stopwords.union(domain_stopwords).union(very_frequent_terms))

print("Number of stopwords:", len(final_stopwords))
print("Data-driven frequent weak terms added:", sorted(list(very_frequent_terms))[:40])


Number of stopwords: 364
Data-driven frequent weak terms added: ['all', 'and', 'any', 'are', 'but', 'can', 'enron', 'for', 'from', 'have', 'not', 'our', 'please', 'thanks', 'that', 'the', 'they', 'this', 'was', 'will', 'with', 'would', 'you', 'your']


## 11. Vectorizers


In [10]:
def build_count_vectorizer():
    return CountVectorizer(
        stop_words=final_stopwords,
        min_df=20,
        max_df=0.25,
        max_features=8000,
        ngram_range=(1, 2),
        token_pattern=r"(?u)\b[a-z_][a-z_]+\b"
    )

def build_tfidf_vectorizer():
    return TfidfVectorizer(
        stop_words=final_stopwords,
        min_df=20,
        max_df=0.25,
        max_features=8000,
        ngram_range=(1, 2),
        token_pattern=r"(?u)\b[a-z_][a-z_]+\b"
    )


## 12. Helper functions for evaluation


In [11]:
def get_topic_words(model, feature_names, n_top_words=15):
    topic_words = {}
    for topic_idx, topic in enumerate(model.components_):
        top_ids = topic.argsort()[:-n_top_words - 1:-1]
        topic_words[topic_idx] = [feature_names[i] for i in top_ids]
    return topic_words

def print_topics(model, feature_names, n_top_words=15):
    topic_words = get_topic_words(model, feature_names, n_top_words=n_top_words)
    for topic_idx, words in topic_words.items():
        print(f"Topic {topic_idx}: " + ", ".join(words))

def compute_topic_diversity(model, feature_names, top_n=15):
    topic_words = get_topic_words(model, feature_names, n_top_words=top_n)
    all_words = [w for words in topic_words.values() for w in words]
    return len(set(all_words)) / len(all_words)

def compute_topic_balance(doc_topic_matrix):
    dominant = np.argmax(doc_topic_matrix, axis=1)
    shares = pd.Series(dominant).value_counts(normalize=True)
    return {
        "largest_topic_share": shares.max(),
        "smallest_topic_share": shares.min(),
        "std_topic_share": shares.std()
    }

def compute_coherence_from_model(model, feature_names, tokenized_docs_for_coherence, top_n=15):
    topic_words = list(get_topic_words(model, feature_names, n_top_words=top_n).values())
    dictionary = Dictionary(tokenized_docs_for_coherence)
    coherence_model = CoherenceModel(
        topics=topic_words,
        texts=tokenized_docs_for_coherence,
        dictionary=dictionary,
        coherence="c_v"
    )
    return coherence_model.get_coherence()

def average_jaccard_topic_stability(topic_lists_a, topic_lists_b):
    # Greedy topic matching across two runs
    used_b = set()
    scores = []
    for words_a in topic_lists_a:
        best_score = -1
        best_j = None
        set_a = set(words_a)
        for j, words_b in enumerate(topic_lists_b):
            if j in used_b:
                continue
            set_b = set(words_b)
            score = len(set_a & set_b) / len(set_a | set_b)
            if score > best_score:
                best_score = score
                best_j = j
        if best_j is not None:
            used_b.add(best_j)
            scores.append(best_score)
    return float(np.mean(scores)) if scores else np.nan

def evaluate_lda(doc_series, tokenized_docs_for_coherence, n_topics, random_state):
    vectorizer = build_count_vectorizer()
    X = vectorizer.fit_transform(doc_series)
    feature_names = vectorizer.get_feature_names_out()

    lda = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=random_state,
        learning_method="batch",
        max_iter=50,
        doc_topic_prior=None,
        topic_word_prior=None,
    )
    doc_topic = lda.fit_transform(X)

    perplexity = lda.perplexity(X)
    coherence = compute_coherence_from_model(lda, feature_names, tokenized_docs_for_coherence, top_n=15)
    diversity = compute_topic_diversity(lda, feature_names, top_n=15)
    balance = compute_topic_balance(doc_topic)

    return {
        "model": lda,
        "vectorizer": vectorizer,
        "X": X,
        "feature_names": feature_names,
        "doc_topic": doc_topic,
        "perplexity": perplexity,
        "coherence_c_v": coherence,
        "topic_diversity": diversity,
        **balance
    }

def evaluate_nmf(doc_series, n_topics, random_state):
    vectorizer = build_tfidf_vectorizer()
    X = vectorizer.fit_transform(doc_series)
    feature_names = vectorizer.get_feature_names_out()

    nmf = NMF(
        n_components=n_topics,
        random_state=random_state,
        init="nndsvda",
        max_iter=500
    )
    doc_topic = nmf.fit_transform(X)
    diversity = compute_topic_diversity(nmf, feature_names, top_n=15)
    balance = compute_topic_balance(doc_topic)

    return {
        "model": nmf,
        "vectorizer": vectorizer,
        "X": X,
        "feature_names": feature_names,
        "doc_topic": doc_topic,
        "topic_diversity": diversity,
        **balance
    }


## 13. Baseline LDA

This baseline is not the final answer.
It is only used to inspect whether the revised corpus is already producing interpretable themes.

In [12]:
baseline = evaluate_lda(
    doc_series=docs_phrased,
    tokenized_docs_for_coherence=tokenized_docs_phrased,
    n_topics=6,
    random_state=RANDOM_STATE
)

print("Baseline DTM shape:", baseline["X"].shape)
print("Perplexity:", round(baseline["perplexity"], 2))
print("Coherence:", round(baseline["coherence_c_v"], 4))
print("Topic diversity:", round(baseline["topic_diversity"], 4))
print_topics(baseline["model"], baseline["feature_names"], n_top_words=15)


Baseline DTM shape: (95480, 8000)
Perplexity: 2581.42
Coherence: 0.5101
Topic diversity: 0.8778
Topic 0: just, time, think, going, good, want, like, work, day, hope, great, make, people, original_message, new
Topic 1: business, group, process, new, time, project, trading, issues, risk, market, provide, information, work, data, team
Topic 2: agreement, mark, credit, enron_north, draft, comments, legal, sara, review, hou_ect, ena, america_corp, enron_north america_corp, phone_fax, letter
Topic 3: gas, deal, price, deals, power, day, contract, let_know, month, capacity, new, enron_communications, pipeline, total, change
Topic 4: meeting, information, contact, received, time, office, houston, available, click, any_questions, following, number, order, friday, monday
Topic 5: energy, power, california, market, state, ferc, iso, company, customers, commission, prices, electricity, order, utilities, new


## 14. Compare candidate LDA models

The number of topics should not be chosen by intuition alone.
We compare several candidate values of `k` and look at:

- coherence,
- diversity,
- balance,
- seed stability,
- qualitative interpretability from representative emails.

In [13]:
candidate_topics = [5, 6, 7, 8]
lda_results = []

for k in candidate_topics:
    out = evaluate_lda(
        doc_series=docs_phrased,
        tokenized_docs_for_coherence=tokenized_docs_phrased,
        n_topics=k,
        random_state=RANDOM_STATE
    )
    lda_results.append({
        "n_topics": k,
        "perplexity": round(out["perple" \
        "xity"], 2),
        "coherence_c_v": round(out["coherence_c_v"], 4),
        "topic_diversity": round(out["topic_diversity"], 4),
        "largest_topic_share": round(out["largest_topic_share"], 4),
        "smallest_topic_share": round(out["smallest_topic_share"], 4),
        "std_topic_share": round(out["std_topic_share"], 4),
    })

lda_results_df = pd.DataFrame(lda_results).sort_values(
    ["coherence_c_v", "topic_diversity", "perplexity"],
    ascending=[False, False, True]
).reset_index(drop=True)

lda_results_df


,n_topics,perplexity,coherence_c_v,topic_diversity,largest_topic_share,smallest_topic_share,std_topic_share
0,8,2435.78,0.5683,0.8583,0.2035,0.0753,0.0487
1,7,2507.74,0.5489,0.8571,0.2233,0.0873,0.0483
2,6,2581.42,0.5101,0.8778,0.2543,0.0933,0.0541
3,5,2638.66,0.4920,0.8933,0.2675,0.1442,0.0504


## 15. Topic stability across random seeds

A good topic solution should not collapse completely when the random seed changes.
This section checks whether the extracted themes remain reasonably similar across multiple runs.

In [16]:
stability_rows = []

for k in candidate_topics:
    run_a = evaluate_lda(docs_phrased, tokenized_docs_phrased, n_topics=k, random_state=42)
    run_b = evaluate_lda(docs_phrased, tokenized_docs_phrased, n_topics=k, random_state=123)
    run_c = evaluate_lda(docs_phrased, tokenized_docs_phrased, n_topics=k, random_state=999)

    words_a = list(get_topic_words(run_a["model"], run_a["feature_names"], 15).values())
    words_b = list(get_topic_words(run_b["model"], run_b["feature_names"], 15).values())
    words_c = list(get_topic_words(run_c["model"], run_c["feature_names"], 15).values())

    stab_ab = average_jaccard_topic_stability(words_a, words_b)
    stab_ac = average_jaccard_topic_stability(words_a, words_c)
    stab_bc = average_jaccard_topic_stability(words_b, words_c)

    stability_rows.append({
        "n_topics": k,
        "stability_mean_jaccard": round(np.mean([stab_ab, stab_ac, stab_bc]), 4)
    })

stability_df = pd.DataFrame(stability_rows).sort_values("stability_mean_jaccard", ascending=False).reset_index(drop=True)
stability_df


,n_topics,stability_mean_jaccard
0,5,0.7932
1,8,0.5219
2,7,0.4498
3,6,0.4232


## 16. NMF as a comparison model

NMF is used here only as a comparison point.
If LDA and NMF point toward a similar thematic structure, that strengthens the argument that the detected themes are not purely accidental.

In [17]:
nmf_results = []

for k in candidate_topics:
    out = evaluate_nmf(docs_phrased, n_topics=k, random_state=RANDOM_STATE)
    nmf_results.append({
        "n_topics": k,
        "topic_diversity": round(out["topic_diversity"], 4),
        "largest_topic_share": round(out["largest_topic_share"], 4),
        "smallest_topic_share": round(out["smallest_topic_share"], 4),
        "std_topic_share": round(out["std_topic_share"], 4),
    })

nmf_results_df = pd.DataFrame(nmf_results).sort_values(
    ["topic_diversity", "std_topic_share"],
    ascending=[False, True]
).reset_index(drop=True)

nmf_results_df


,n_topics,topic_diversity,largest_topic_share,smallest_topic_share,std_topic_share
0,6,0.9889,0.2906,0.0081,0.1173
1,5,0.9733,0.4382,0.0084,0.1737
2,7,0.9524,0.2812,0.0068,0.0986
3,8,0.9417,0.2750,0.0068,0.0895


## 17. Model-selection decision

The automatic ranking selected **8 topics** because this solution achieved the best coherence and perplexity among the tested LDA models.

However, the final model was set to **5 topics** after reviewing the full evidence. The 5-topic solution showed the **highest topic stability across random seeds** and the **highest topic diversity**, while also producing broader and more interpretable thematic groups.

This distinction matters in this project. The goal is not to maximize the number of narrowly separated clusters, but to identify the **main communication themes** in the Enron email corpus. For that reason, interpretability and stability were prioritized over a moderate gain in coherence and perplexity.

Therefore, the final decision was to use **5 topics** for interpretation.

In [18]:
selection_df = (
    lda_results_df.merge(stability_df, on="n_topics", how="left")
    .copy()
)

# Higher is better for coherence, diversity, stability.
# Lower is better for perplexity and topic-share imbalance.
selection_df["perplexity_rank"] = selection_df["perplexity"].rank(method="min", ascending=True)
selection_df["coherence_rank"] = selection_df["coherence_c_v"].rank(method="min", ascending=False)
selection_df["diversity_rank"] = selection_df["topic_diversity"].rank(method="min", ascending=False)
selection_df["stability_rank"] = selection_df["stability_mean_jaccard"].rank(method="min", ascending=False)
selection_df["balance_rank"] = selection_df["std_topic_share"].rank(method="min", ascending=True)

selection_df["combined_rank_score"] = (
    0.35 * selection_df["coherence_rank"] +
    0.20 * selection_df["diversity_rank"] +
    0.20 * selection_df["stability_rank"] +
    0.15 * selection_df["balance_rank"] +
    0.10 * selection_df["perplexity_rank"]
)

selection_df = selection_df.sort_values("combined_rank_score", ascending=True).reset_index(drop=True)
selection_df

,n_topics,perplexity,coherence_c_v,topic_diversity,largest_topic_share,smallest_topic_share,std_topic_share,stability_mean_jaccard,perplexity_rank,coherence_rank,diversity_rank,stability_rank,balance_rank,combined_rank_score
0,8,2435.78,0.5683,0.8583,0.2035,0.0753,0.0487,0.5219,1.0,1.0,3.0,2.0,2.0,1.75
1,7,2507.74,0.5489,0.8571,0.2233,0.0873,0.0483,0.4498,2.0,2.0,4.0,3.0,1.0,2.45
2,5,2638.66,0.4920,0.8933,0.2675,0.1442,0.0504,0.7932,4.0,4.0,1.0,1.0,3.0,2.65
3,6,2581.42,0.5101,0.8778,0.2543,0.0933,0.0541,0.4232,3.0,3.0,2.0,4.0,4.0,3.15


In [20]:
# Automatic recommendation based on the combined ranking above
AUTO_FINAL_K = int(selection_df.loc[0, "n_topics"])

# Manual override is allowed after reviewing representative emails.
# If the automatic choice is not the most interpretable one, replace it manually.
FINAL_K = 5

print("AUTO_FINAL_K =", AUTO_FINAL_K)
print("FINAL_K used for the final model =", FINAL_K)

final_eval = evaluate_lda(
    doc_series=docs_phrased,
    tokenized_docs_for_coherence=tokenized_docs_phrased,
    n_topics=FINAL_K,
    random_state=RANDOM_STATE
)

print("\nFINAL MODEL METRICS")
print("-" * 60)
print("n_topics:", FINAL_K)
print("Perplexity:", round(final_eval["perplexity"], 2))
print("Coherence:", round(final_eval["coherence_c_v"], 4))
print("Topic diversity:", round(final_eval["topic_diversity"], 4))
print("Largest topic share:", round(final_eval["largest_topic_share"], 4))
print("Smallest topic share:", round(final_eval["smallest_topic_share"], 4))
print("Std topic share:", round(final_eval["std_topic_share"], 4))
print()
print_topics(final_eval["model"], final_eval["feature_names"], n_top_words=18)

AUTO_FINAL_K = 8
FINAL_K used for the final model = 5

FINAL MODEL METRICS
------------------------------------------------------------
n_topics: 5
Perplexity: 2638.66
Coherence: 0.492
Topic diversity: 0.8933
Largest topic share: 0.2675
Smallest topic share: 0.1442
Std topic share: 0.0504

Topic 0: just, time, think, going, want, good, like, work, day, hope, great, people, make, original_message, did, let_know, week, new
Topic 1: power, market, energy, california, state, ferc, company, customers, iso, business, new, year, issues, order, million, commission, prices, issue
Topic 2: agreement, mark, credit, legal, enron_north, draft, review, comments, hou_ect, sara, ena, let_know, america_corp, enron_north america_corp, phone_fax, letter, doc, send
Topic 3: gas, deal, price, deals, day, contract, power, let_know, month, new, capacity, change, data, original_message, enron_communications, total, daily, ena
Topic 4: meeting, information, contact, time, new, group, business, received, office

## 18. Representative emails for the selected final model

This section is the main qualitative validation step.

Topic labels should be assigned only after comparing:
1. the top words of each topic,
2. the most representative emails,
3. the prevalence of each topic in the corpus,
4. the overall interpretability of the full solution.

A topic should not be labeled from keywords alone. In email corpora, high-frequency words may still hide residual template noise, generic office communication, or repeated document families. For that reason, representative emails are necessary before making final thematic claims.

In [21]:
doc_topic_final = final_eval["doc_topic"]
topic_words_final = get_topic_words(final_eval["model"], final_eval["feature_names"], n_top_words=12)

N_EXAMPLES = 3

for topic_idx in range(FINAL_K):
    print("=" * 140)
    print(f"TOPIC {topic_idx}")
    print("=" * 140)
    print("Top words:", ", ".join(topic_words_final[topic_idx]))
    print()

    top_doc_indices = np.argsort(doc_topic_final[:, topic_idx])[::-1][:N_EXAMPLES]

    for rank, doc_idx in enumerate(top_doc_indices, start=1):
        score = doc_topic_final[doc_idx, topic_idx]
        print(f"[Representative email #{rank}] document_position={doc_idx} topic_score={score:.4f}")
        print(docs_raw.iloc[doc_idx][:2200])
        print("\n" + "-" * 140 + "\n")


TOPIC 0
Top words: just, time, think, going, want, good, like, work, day, hope, great, people

[Representative email #1] document_position=73176 topic_score=0.9969
what time is it am name that appears on your birth certificate susan margaret scott nicknames sue susie susie q subaru suebob the booz brother in law parent s names chuck sally number of candles that appeared on your last cake can t remember date that you regularly blow them out january usually super bowl weekend pets dog bailey at my parents house height eye color blue piercing just one in each ear hair color blonde tattoos none and there never will be any how much do you love your job a heck of a lot more than my last one of course any time you go from hour days to hour days it s got to be a good thing hometown houston current residence houston have you been in love never currently in love see above favorite thing about email keeping in touch with friends and relatives who ve moved away and getting hold of the people in ho

## 19. Topic labels and interpretation table

The labels below were assigned after reviewing both the top words and the most representative emails for each topic in the selected final model.

Because topic IDs are specific to the fitted model, the interpretation presented in this table applies to the current final solution with **5 topics**.

In [27]:
# Final topic labels for the selected 5-topic model
topic_labels = {
    0: "Informal and personal employee communication",
    1: "California energy market and regulatory affairs",
    2: "Legal, credit, and contract documentation",
    3: "Trading activity and deal-flow reporting",
    4: "Internal operations, training, and corporate coordination"
}

topic_summary_rows = []
for topic_idx in range(FINAL_K):
    topic_summary_rows.append({
        "topic_id": topic_idx,
        "label": topic_labels.get(topic_idx, "???"),
        "top_words": ", ".join(topic_words_final[topic_idx][:12])
    })

topic_summary_df = pd.DataFrame(topic_summary_rows)
topic_summary_df

,topic_id,label,top_words
0,0,Informal and personal employee communication,"just, time, think, going, want, good, like, work, day, hope, great, people"
1,1,California energy market and regulatory affairs,"power, market, energy, california, state, ferc, company, customers, iso, business, new, year"
2,2,"Legal, credit, and contract documentation","agreement, mark, credit, legal, enron_north, draft, review, comments, hou_ect, sara, ena, let_know"
3,3,Trading activity and deal-flow reporting,"gas, deal, price, deals, day, contract, power, let_know, month, new, capacity, change"
4,4,"Internal operations, training, and corporate coordination","meeting, information, contact, time, new, group, business, received, office, houston, team, available"


## 20. Topic prevalence in the corpus

This table reports how often each topic appears as the dominant topic in the corpus.

The purpose of this step is to show whether the final model is dominated by one very large topic or whether the thematic structure is distributed more broadly across several major communication themes.

In [28]:
dominant_topics = np.argmax(doc_topic_final, axis=1)

topic_distribution = (
    pd.Series(dominant_topics)
    .value_counts()
    .sort_index()
    .rename_axis("topic_id")
    .reset_index(name="document_count")
)

topic_distribution["share_percent"] = (
    topic_distribution["document_count"] / len(docs_phrased) * 100
).round(2)

topic_distribution["label"] = topic_distribution["topic_id"].map(topic_labels)

topic_distribution = topic_distribution[
    ["topic_id", "label", "document_count", "share_percent"]
]

topic_distribution

,topic_id,label,document_count,share_percent
0,0,Informal and personal employee communication,25541,26.75
1,1,California energy market and regulatory affairs,13764,14.42
2,2,"Legal, credit, and contract documentation",20667,21.65
3,3,Trading activity and deal-flow reporting,14877,15.58
4,4,"Internal operations, training, and corporate coordination",20631,21.61


### Interpretation of topic prevalence

The final 5-topic solution is relatively balanced: no topic dominates the corpus completely, and all five topics account for a meaningful share of documents.

At the same time, the prevalence table shows that the corpus is not limited to external market or trading content. A substantial portion of the emails belongs to broader internal communication categories, including informal exchange, corporate coordination, and operational information.

This is an important result rather than a problem. It suggests that the Enron email corpus captures not only market-facing activity, but also the internal social, legal, and administrative communication that supported the company’s daily operations.

## 21. Final written interpretation

**Selected number of topics:** `5`

The final model was set to 5 topics because this solution provided the most interpretable overall summary of the corpus. Although the 8-topic solution achieved higher coherence and lower perplexity, the 5-topic solution showed substantially higher stability across random seeds and higher topic diversity. In addition, the 5-topic structure was easier to interpret as a set of broad and meaningful communication themes rather than narrower sub-clusters.

**Final topic interpretation:**  

- **Topic 0 — Informal and personal employee communication**  
  This topic captures casual interpersonal exchange, including social conversation, personal updates, and other non-technical employee communication.

- **Topic 1 — California energy market and regulatory affairs**  
  This topic reflects discussion of the electricity market, California energy issues, FERC activity, and broader regulatory concerns.

- **Topic 2 — Legal, credit, and contract documentation**  
  This topic is centered on agreements, legal review, draft documents, credit-related communication, and contract-processing workflows.

- **Topic 3 — Trading activity and deal-flow reporting**  
  This topic captures operational discussion of gas and power deals, prices, contracts, transaction volumes, and market activity reporting.

- **Topic 4 — Internal operations, training, and corporate coordination**  
  This topic groups together internal announcements, training information, website rollout communication, IT maintenance updates, and other coordination-oriented messages.

**Overall conclusion:**  

The final 5-topic model provides a clear and defensible summary of the main communication themes in the cleaned Enron email corpus. The results show that employee communication was not limited to trading and market issues. It also included substantial legal-document flow, internal operational coordination, and informal interpersonal exchange. This makes the final interpretation broader, but also more realistic for a large corporate email dataset.


## 22. Limitations

Even after stronger cleaning, the Enron corpus remains highly heterogeneous. Some emails combine several subjects in the same message, and some repeated internal communication families may still survive the filtering process.

In the final 5-topic solution, Topic 0 is especially broad and reflects general informal employee communication rather than a narrowly defined business process. Topic 4 is also somewhat mixed, combining training, operational announcements, system maintenance, and internal coordination.

Therefore, the correct interpretation is not that the notebook recovers perfectly pure ground-truth topics. Rather, it produces a defensible and interpretable approximation of the dominant communication themes in the corpus.



## 23. References

Blei, D. M., Ng, A. Y. and Jordan, M. I. (2003) ‘Latent Dirichlet Allocation’, *Journal of Machine Learning Research*, 3, pp. 993–1022.

Chang, J., Boyd-Graber, J., Gerrish, S., Wang, C. and Blei, D. M. (2009) ‘Reading Tea Leaves: How Humans Interpret Topic Models’, *Advances in Neural Information Processing Systems*, 22.

Cohen, W. W. (2015) *Enron Email Dataset*. Carnegie Mellon University.

Pedregosa, F. et al. (2011) ‘Scikit-learn: Machine Learning in Python’, *Journal of Machine Learning Research*, 12, pp. 2825–2830.

Röder, M., Both, A. and Hinneburg, A. (2015) ‘Exploring the Space of Topic Coherence Measures’, *Proceedings of the Eighth ACM International Conference on Web Search and Data Mining*, pp. 399–408.

Lee, D. D. and Seung, H. S. (1999) ‘Learning the Parts of Objects by Non-negative Matrix Factorization’, *Nature*, 401, pp. 788–791.